# 第 34 课：AEC 回声消除——参考信号、NLMS 与 Double-talk

扬声器播放的远端声音会经房间和设备耦合进入麦克风。AEC 使用“扬声器参考信号”估计这条回声路径。

<!-- course-upgrade-v2 -->
## 学习导航

| 项目 | 内容 |
|---|---|
| 所属阶段 | 音频信号前端 |
| 建议投入 | 3～5 小时，可分 2～3 次完成 |
| 前置要求 | 完成第 33 课；如果前测低于 2/3，先回看上一课小结 |
| 本课核心 | AEC reference、NLMS、double-talk |
| 完成标准 | 能口头解释核心概念；独立完成强化题；从空白重写核心函数 |

高效顺序：**先回答前测 → 预测代码结果 → 再运行 → 修改一个变量 → 关闭答案复现 → 次日回忆。**


<!-- course-upgrade-v2 -->
## 课前诊断（先不要运行代码）

1. 分别用一句话解释：AEC reference、NLMS、double-talk。
2. 画出这三个概念之间的输入—输出关系。
3. 写下你最不确定的一点，并给出一个暂时猜测。

自评：答对 0～1 题先复习前置课；答对 2 题可以正常学习；3 题都能讲清楚则直接挑战代码和迁移题。


<!-- course-bridge-v3 -->
## 知识接力：先取回旧知识，再进入本课

### 3 分钟闭卷回忆

在新 Markdown cell 中回答，**不要先翻前文**：采样率、PCM、通道、RMS/dBFS、SNR；流式 chunk/cache 与时间戳。

- 三项都能用“含义 + 单位/shape + 一个数字例子”回答：进入本课。
- 能回答两项：学习本课，但把缺口记入 `LEARNING_LOG.md`。
- 只能回答零到一项：先回到 [上一课](33_VAD与Endpoint_起点终点和Hangover.ipynb)与[唯一学习路径](../LEARNING_PATH.md)，做一次最小实验；不要靠继续看新术语掩盖断点。

### 本课接口契约

```text
输入：真实麦克风波形、参考信号/多通道和会话状态
  ↓ 本课要学会的变换、状态或判断
输出：保持时间映射、可旁路、可 reset 且用 CER/SNR 双重验证的前端输出
```

学完后必须能解释：输入的哪个单位/shape/状态若丢失，会让输出“仍能运行却语义错误”。


In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import soundfile as sf
import librosa

def find_root():
    here=Path.cwd().resolve()
    for p in [here,*here.parents]:
        if (p/"pyproject.toml").exists():return p
    raise FileNotFoundError("请从 learn_asr 或 notebooks 目录启动 Jupyter")
ROOT=find_root();plt.rcParams["figure.figsize"]=(11,4)
print("项目根目录:",ROOT)

from scipy.signal import lfilter
rng=np.random.default_rng(3);sr=8000;t=np.arange(sr*2)/sr
far=(.5*np.sin(2*np.pi*260*t)+.25*np.sin(2*np.pi*420*t)).astype(np.float32)
echo_path=np.array([0,0,0,.7,.35,.18,.08],np.float32);echo=lfilter(echo_path,[1],far)
near=np.zeros_like(far);speech,_=sf.read(ROOT/"data"/"spoken_digits_parts"/"8_jackson_0.wav");speech=speech.astype(np.float32)
near[5000:5000+min(len(speech),len(near)-5000)]=speech[:len(near)-5000];mic=echo+near+.01*rng.normal(size=len(far))

## 1. NLMS 自适应滤波器

In [ ]:
def nlms(reference,mic,taps=32,mu=.5,eps=1e-6):
    w=np.zeros(taps);out=np.zeros_like(mic);estimated=np.zeros_like(mic)
    padded=np.pad(reference,(taps-1,0))
    for n in range(len(mic)):
        x=padded[n:n+taps][::-1];yhat=np.dot(w,x);e=mic[n]-yhat
        w+=mu*e*x/(np.dot(x,x)+eps);estimated[n]=yhat;out[n]=e
    return out,estimated,w
cleaned,estimated,w=nlms(far,mic)
fig,ax=plt.subplots(3,1,figsize=(11,6),sharex=True)
for a,z,title in zip(ax,[mic,estimated,cleaned],["microphone","estimated echo","AEC output"]):a.plot(t,z);a.set_title(title)
plt.tight_layout();plt.show()

## 2. ERLE 衡量回声衰减

只应在 near-end 不说话的区段评估 echo return loss enhancement。

In [ ]:
region=slice(1000,4500);erle=10*np.log10(np.mean(mic[region]**2)/np.mean(cleaned[region]**2));print("ERLE dB",erle)

## 3. Double-talk 是难点

近端用户和远端扬声器同时说话时，NLMS 会把近端语音误当成回声误差并污染滤波器。真实 AEC 需要 double-talk detection、nonlinear processing、延迟估计和时钟漂移处理。

## 本课测试

1. AEC 为什么需要 far-end reference？
2. 固定降噪能否替代 AEC？
3. NLMS 的 filter taps 表示什么？
4. ERLE 应在哪类区段测？
5. double-talk 时为什么要降低/冻结更新？

<details><summary>展开参考答案</summary>

1. 用于预测扬声器回声。2. 不能，回声与参考高度相关且随路径变化。3. 所建模回声路径长度。4. 无近端语音的回声区段。5. 防止把近端语音学进回声滤波器。

</details>

<!-- course-upgrade-v2 -->
## 强化练习：第 34 课专属题库

请先把答案写进新的 Markdown/Code cell，再展开自评标准。

### A. 基础回忆

1. 不看上文，分别定义 `AEC reference`、`NLMS`、`double-talk`。
2. 哪一个量/状态是本课最容易在模块边界丢失的？它的单位和 shape 是什么？
3. 本课至少写出两个“看起来能运行，但结果其实错误”的例子。

### B. 预测与推理

4. 场景：**近端说话时滤波器继续快速更新**。先预测现象，再说明原因，最后给出一项可以验证猜测的指标。
5. 改变本课最关键参数的 0.5×、1×、2×，分别预测准确率、延迟、内存或数值误差怎样变化。
6. 画一张最小数据流图，在每条边标出 dtype、shape、时间单位或概率/代价方向。

### C. 编程与排错

7. 编程任务：**实现冻结更新的 double-talk 基线**。至少加入正常、边界、错误输入三类测试。
8. 故意制造一个 off-by-one、shape、状态未 reset 或数值稳定性错误；记录错误现象和定位过程。
9. 不看本课实现，从空白 cell 重写最核心函数，并用原实现作数值对照。

### D. 迁移与表达

10. 跨课任务：**说明 AEC 为什么必须在模型前端早期**。
11. 用 90 秒向没有学过 ASR 的人解释本课；禁止只念术语，必须举一个数字或生活例子。
12. 写出一个生产系统中会监控的指标，以及它异常时优先检查的三处位置。

<details><summary>展开自评标准</summary>

- 每题 0～2 分：0=无法回答；1=方向正确但缺少单位、边界或验证；2=解释完整且能用代码/数字验证。
- 24 分满分：达到 19 分再进入下一课；15～18 分次日重做错题；低于 15 分回看本课图和核心代码。
- 第 4 题必须包含“预测—原因—指标”，第 7～9 题必须真正运行测试，第 10 题必须明确上下游 contract。
- 核心答案至少应正确使用：AEC reference、NLMS、double-talk。

</details>


<!-- course-upgrade-v2 -->
## 间隔复习与离场票

### 离场票（现在完成）

- [ ] 我能不用笔记解释 AEC reference、NLMS、double-talk。
- [ ] 我能说出本课最常见的错误及其观测现象。
- [ ] 我能从空白重写一个核心函数，并通过至少 3 个测试。
- [ ] 我能说明本课对上一层和下一层接口的影响。

### 复习时间表

- **明天（5 分钟）**：闭卷写出三个核心概念和一个公式/shape。
- **7 天后（15 分钟）**：重做第 4、7、10 题，不运行原答案。
- **30 天后（20 分钟）**：从真实音频或随机张量重新构造一个最小实验。

把错题记录到根目录 `LEARNING_LOG.md`。不要只写“不会”，要写：原判断、证据、正确规则、下次检查动作。
